In [7]:
import pandas as pd
import numpy as np

#Load file
rot_df = pd.read_csv("SPOC_ls_TRUE_power_and_period.csv")
master_df = pd.read_csv("ASTR502_Mega_Target_List.csv")


#Fix TIC formatting between mega list and our csv

# Extract numeric TICID safely
master_df["ticid"] = (
    master_df["tic_id"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
)

# Convert to numeric (invalid → NaN)
master_df["ticid"] = pd.to_numeric(master_df["ticid"], errors="coerce")

# Drop bad rows BEFORE converting to int
master_df = master_df.dropna(subset=["ticid"])

# Now safe to convert
master_df["ticid"] = master_df["ticid"].astype(int)


# Rotation file (also make safe)
rot_df["ticid"] = pd.to_numeric(rot_df["ticid"], errors="coerce")
rot_df = rot_df.dropna(subset=["ticid"])
rot_df["ticid"] = rot_df["ticid"].astype(int)


#Clean Nans

rot_df = rot_df.dropna(subset=["true_period_d"])
master_df = master_df.dropna(subset=["pl_orbper"])

print("\n--- TICID CLEANING CHECK ---")
print(f"Master TICIDs: {master_df['ticid'].nunique()}")
print(f"Rotation TICIDs: {rot_df['ticid'].nunique()}")

common = set(master_df["ticid"]).intersection(set(rot_df["ticid"]))
print(f"Overlapping TICIDs: {len(common)}")

# MANY-TO-MANY MERGE on TICID
merged = pd.merge(rot_df, master_df, on="ticid", how="inner")

print(f"\nMerged rows: {len(merged)}")

#Find period ratios between orbital and true period from our csv
merged["ratio"] = merged["true_period_d"] / merged["pl_orbper"]

# Harmonic checks (10% tolerance)
merged["match_1x"] = np.abs(merged["ratio"] - 1.0) < 0.10
merged["match_2x"] = np.abs(merged["ratio"] - 2.0) < 0.10
merged["match_half"] = np.abs(merged["ratio"] - 0.5) < 0.10

# Combined match flag
merged["is_match"] = (
    merged["match_1x"] |
    merged["match_2x"] |
    merged["match_half"]
)

#Match the type
def classify_match(row):
    if row["match_1x"]:
        return "1x (orbital)"
    elif row["match_2x"]:
        return "2x (rotation ≈ 2× orbital)"
    elif row["match_half"]:
        return "0.5x (rotation ≈ 0.5× orbital)"
    else:
        return "no match"

merged["match_type"] = merged.apply(classify_match, axis=1)

#Filter matches
matches = merged[merged["is_match"]].copy()

print(f"Total raw matches (including duplicates): {len(matches)}")

# Keep best match per TICID + sector
matches["abs_diff_from_1"] = np.abs(matches["ratio"] - 1.0)

matches = matches.sort_values("abs_diff_from_1")
matches = matches.drop_duplicates(subset=["ticid", "sector"], keep="first")

print(f"Unique sector matches: {len(matches)}")

#Save flagged sectors
output_cols = [
    "ticid",
    "sector",
    "cadence",
    "true_period_d",
    "pl_orbper",
    "ratio",
    "match_type"
]

matches[output_cols].to_csv("flagged_transit_matches.csv", index=False)

#Stats

# Total sectors analyzed
total_sectors = len(rot_df)

# Unique TICIDs in rotation sample
total_tics = rot_df["ticid"].nunique()

# Flagged sectors
flagged_sectors = len(matches)

# Unique TICIDs affected
flagged_tics = matches["ticid"].nunique()

# Fractions
sector_contamination_frac = flagged_sectors / total_sectors
tic_contamination_frac = flagged_tics / total_tics

# Per-TIC sector counts
sectors_per_tic = matches.groupby("ticid")["sector"].nunique()
sectors_per_tic = sectors_per_tic.sort_values(ascending=False)

sectors_per_tic.to_csv("sectors_per_tic_flagged.csv")

# Match type breakdown
match_breakdown = matches["match_type"].value_counts()

# Print summary
print("\nSUMMARY")

print("\nSECTOR LEVEL")
print(f"Total sectors analyzed: {total_sectors}")
print(f"Flagged sectors: {flagged_sectors}")
print(f"Fraction contaminated: {sector_contamination_frac:.4f}")

print("\n--- TIC LEVEL ---")
print(f"Total TICIDs: {total_tics}")
print(f"TICIDs with contamination: {flagged_tics}")
print(f"Fraction of TICIDs affected: {tic_contamination_frac:.4f}")

print("\nSECTORS PER TIC (flagged only)")
print(sectors_per_tic.describe())

print("\nMATCH TYPE BREAKDOWN")
print(match_breakdown)

# -----------------------------
# Sanity check: overlap
# -----------------------------
common_tics = set(rot_df["ticid"]).intersection(set(master_df["ticid"]))
print(f"\nNumber of overlapping TICIDs: {len(common_tics)}")


--- TICID CLEANING CHECK ---
Master TICIDs: 3423
Rotation TICIDs: 680
Overlapping TICIDs: 678

Merged rows: 2086
Total raw matches (including duplicates): 626
Unique sector matches: 537

SUMMARY

SECTOR LEVEL
Total sectors analyzed: 1643
Flagged sectors: 537
Fraction contaminated: 0.3268

--- TIC LEVEL ---
Total TICIDs: 680
TICIDs with contamination: 241
Fraction of TICIDs affected: 0.3544

SECTORS PER TIC (flagged only)
count    241.000000
mean       2.228216
std        2.731337
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       22.000000
Name: sector, dtype: float64

MATCH TYPE BREAKDOWN
match_type
1x (orbital)                      285
0.5x (rotation ≈ 0.5× orbital)    199
2x (rotation ≈ 2× orbital)         53
Name: count, dtype: int64

Number of overlapping TICIDs: 678
